# Lab04: Genetic Algorithms 

# About next week seminar

## Recap

## Overview of the lab

In this lab, we will explore genetic algorithms. Genetic algorithms (GAs) are optimization and search techniques in artificial intelligence inspired by the process of natural evolution. They are used to find good or near-optimal solutions to complex problems where traditional methods may be too slow or infeasible.

## 1. Introductionn to Genetic algorithms

* A ***search and optimization technique*** inspired by natural evolution.
* Uses the principles of selection, crossover, and mutation.
* Evolves a population of possible solutions to find the best (fittest) one.

Each solution (often called an individual or chromosome) is represented by a set of parameters - similar to genes in biology. The fitness of each individual measures how well it solves the problem at hand.

Key Concepts:

* ***Gene*** → A single parameter or feature that defines part of a solution.
(Example: weight, value, or decision variable)

* ***Chromosome*** → A complete set of genes; represents one potential solution.
(Example: [x₁, x₂, x₃, …])

* ***Population*** → A group of chromosomes; represents many solutions being tested at once.

![image.png](images/GeneticAlgorithmDefinitions.png)

## 2.  Main steps in Genetic Algorithms 
Initialization (Population) $\rightarrow$ Selection (Fitness Function) $\rightarrow$ Crossover $\rightarrow$ Mutation

![ga_scheme](images/genetic_algorithm_loop.png)

#### Example:

![ga_scheme](images/GeneticAlgorithmExample.png)

## 2.1. Selection
Selection is the stage of a genetic algorithm in which individual chromosomes are chosen from a population for later breeding (using the crossover operator).

In genetic algorithms (GAs), ***selection pressure*** refers to how strongly the algorithm favors the best-performing individuals (solutions) when choosing who gets to reproduce and create the next generation.

*Selection pressure* is the degree of bias towards fitter individuals in the selection process.
* ***High selection pressure*** → strongly favors the best individuals
   * → faster convergence, but risk of premature convergence (loss of diversity, stuck in local optima).
* ***Low selection pressure*** → individuals are chosen more randomly
   * → slower convergence, but better exploration of the search space (more genetic diversity).

## Roulette Wheel

Chromosomes are selected with a probability proportional to their fitness. Compute each individual’s selection **probability = (individual fitness / total fitness)**

Imagine a roulette wheel where each chromosome has a slot size corresponding to its fitness value; fitter individuals have larger slots and are more likely to be selected.

#### Example:
![image.png](images/RouletteWheelSelection.png)


|             Pros                 | Cons                                                                                         |
|  ------------------------------- | ------------------------                                                                     | 
|  Simple                          | Premature convergence if a single individual is overwhelmingly fit (takes over the wheel).   |
| Works well when fitness differences are moderate. | Sensitive to scaling: small differences in fitness can have a large effect. |

1. Calculate the total fitness of the population.
2. Compute each individual’s selection probability = (individual fitness / total fitness).
3. Spin the wheel (generate a random number) to select individuals based on these probabilities.

In [94]:
import numpy as np

population = ['A', 'B', 'C', 'D', 'E']
fitness_scores = [10, 20, 30, 25, 15] #higher is better

In [95]:
def roulette_wheel_selection(population, fitness_scores, num_selections):
   total_fitness = sum(fitness_scores)
   selection_probs = [f / total_fitness for f in fitness_scores]
   selected_chromosomes = np.random.choice(population, size=num_selections, replace=True, p=selection_probs)
   return selected_chromosomes

selected = roulette_wheel_selection(population, fitness_scores, 3)
print("Selected chromosomes:", selected)

Selected chromosomes: ['C' 'C' 'D']


`np.random.choice` is used to randomly select elements from an array (or a range of integers).
It can sample with or without replacement and can also use probabilities for weighted selection. 

More in [NumPy_notes](/00_NumPy_notes.ipynb)

the argument `replace=True` controls whether the same element can be chosen more than once.

* `replace=True` → sampling with replacement

   * The same chromosome (individual from the population) can be selected multiple times.
   * After one is chosen, it’s still "available" for future selections.
   * This is common in genetic algorithms, because the most fit individuals can be selected more than once (proportional to their selection probability).

* `replace=False` → sampling without replacement
  * Once an element is chosen, it cannot be selected again.
  * This ensures all selected chromosomes are unique.

## Tournament Selection
Randomly select a small group (a “tournament”) of individuals and pick the best one from that group to be a parent.
  * Selection pressure can be controlled by changing 𝑘.
      * Larger $k$ → stronger selection pressure (more elitist).
      * Smaller 𝑘: → more diversity.

|             Pros                 | Cons                                                                                         |
|  ------------------------------- | ------------------------                                                                     | 
|  Easy to implement. Selection pressure can be controlled by changing 𝑘.  | May lose diversity if tournament size is too large.  |

In [ ]:
#Implementation of Tournament Selection with indexes
def tournament_selection(population, fitness_scores, tournament_size, num_selections):
   selected_chromosomes = []
   for _ in range(num_selections):
        #choose random indexes of chromosomes from the population for the tournament
        tournament = np.random.choice(len(population), tournament_size, replace=False)

        #chromosomes in the tournament
        #selected_population = [population[i] for i in tournament]
        #print("Tournament participants:", selected_population)
        #print("Fitness scores:", [fitness_scores[i] for i in tournament])

        #select the index of best chromosome from the tournament
        winner = tournament[np.argmax([fitness_scores[i] for i in tournament])]
        #print("Tournament winner:", population[winner])
        selected_chromosomes.append(population[winner])

   return selected_chromosomes

selected = tournament_selection(population, fitness_scores, tournament_size=3, num_selections=3)
print("Selected:", selected)

Selected: ['C', 'B', 'B']


## Rank-Based Selection

Instead of using raw fitness values, individuals are ranked according to rank. The selection probability depends on rank, not the absolute fitness.

|             Pros                 | Cons                                                                                         |
|  ------------------------------- | ------------------------                                                                     | 
|  Prevents domination by one very fit individual  | Can be slower to converge because it reduces differences between individuals |
| Maintains steady selection pressure  |                                                                                          |

1. Sort the population by fitness (best to worst).
2. Assign selection probabilities based on rank (e.g., linearly or exponentially decreasing).
3. Select individuals using these rank-based probabilities.

In [97]:
def rank_based_selection(population, fitness_scores, num_selections):
   # Rank individuals based on fitness
    sorted_indices = np.argsort(fitness_scores)
    sorted_fitness = [(i, fitness_scores[i], population[i]) for i in sorted_indices]
    print("Sorted indices and their scores:", sorted_fitness)

    ranks = np.arange(1, len(fitness_scores)+1)
    print("Ranks:", ranks)

    probs = ranks / np.sum(ranks)
    print("Selection probabilities:", probs)
    print("Elements by rank: ", np.array(population)[sorted_indices])

    chosen = np.random.choice(np.array(population)[sorted_indices], size=num_selections, p=probs)
    return chosen

selected = rank_based_selection(population, fitness_scores, num_selections=3)
print("Selected chromosomes:", selected)

Sorted indices and their scores: [(np.int64(0), 10, 'A'), (np.int64(4), 15, 'E'), (np.int64(1), 20, 'B'), (np.int64(3), 25, 'D'), (np.int64(2), 30, 'C')]
Ranks: [1 2 3 4 5]
Selection probabilities: [0.06666667 0.13333333 0.2        0.26666667 0.33333333]
Elements by rank:  ['A' 'E' 'B' 'D' 'C']
Selected chromosomes: ['D' 'B' 'B']


`numpy.arange([start,] stop[, step,] dtype=None)`

It creates an array with evenly spaced values within a given range - similar to Python’s built-in range(), but returns a NumPy array instead of a list.
 * ***Python list:*** Can store mixed data types in the same list.
 * ***NumPy array:*** All elements must have the same data type. NumPy will try to cast them to a common type.

 More information in [NumPy_notes](/00_NumPy_notes.ipynb)

## Stochastic Universal Sampling (SUS)

A variation of roulette wheel selection that ensures a more even and consistent spread of selected individuals.

Imagine the roulette wheel as before, but instead of one spin, place multiple evenly spaced pointers on it (equal to the number of parents needed).
   * Spin the wheel once - all pointers select individuals simultaneously.
   
|             Pros                 | Cons                                                                                         |
|  ------------------------------- | ------------------------                                                                     | 
| Reduces randomness in selection (less variance)  | Slightly more complex to implement than standard roulette selection.         |
| Ensures even sampling proportional to fitness.  |                                                                               |

#### Example:
![image.png](images/SUSSelection.png)

## Elitism
The ***best individuals from the current generation*** are automatically passed to the next generation, without alteration.
  
|             Pros                 | Cons                                                                                         |
|  ------------------------------- | ------------------------                                                                     | 
| Guarantees that the best solutions are preserved.  | Reduces diversity         |
| Improves convergence speed.  |                                                                               |


## Truncation Selection
Select only the top fraction of the population (e.g., top 30%) based on fitness. All selected individuals have equal probability of being parents.
  1. Sort population by fitness.
  2. Keep only the top percentage (say, best 20–50%).
  3. Randomly select parents from this truncated set.

|             Pros                 | Cons                                                                                         |
|  ------------------------------- | ------------------------                                                                     | 
| Very simple and deterministic.  | Can quickly reduce genetic diversity.      |
| Strong selection pressure (fast convergence).  |   May get stuck in local optima.


### Summary of Selection Methods

| Method             | Key Idea                        | Strength                 | Weakness               |
| ------------------ | ------------------------------- | ------------------------ | ---------------------- |
| **Roulette Wheel** | Prob. proportional to fitness   | Simple                   | Premature convergence  |
| **Tournament**     | Best of random subset wins      | Adjustable pressure      | Can lose diversity     |
| **Rank-Based**     | Selection by rank               | Stable pressure          | Slower convergence     |
| **SUS**            | Evenly spaced roulette sampling | Low variance             | More complex           |
| **Elitism**        | Best always survive             | Guarantees progress      | Less diversity         |
| **Truncation**     | Top % survive                   | Simple & strong pressure | Fast loss of diversity |

## 2.2 Crossover

Crossover is the process of combining two or more parent solutions to produce a new child solution. It is one of the core mechanisms of a genetic algorithm, allowing promising traits from existing solutions to be combined to create potentially better ones. The choice of crossover method can significantly influence the algorithm's performance and its ability to find good solutions.


## One/Two Point Crossover

We sample one/two points at random, and swap the contents of the corresponding arrays


![crossover](images/crossovers.png)

In [98]:
def one_point_crossover(first,second):
    point = np.random.randint(1,len(first))
    #print("Point",point)
    return first[:point] + second[point:], second[:point] + first[point:]

def one_point_crossover_no_duplicates(first, second):
    point = np.random.randint(1, len(first))
    #print("Point",point)

    child1 = first[:point]
    child2 = second[:point]

    # Fill remaining genes while preserving order and avoiding duplicates
    child1 += [gene for gene in second if gene not in child1]
    child2 += [gene for gene in first if gene not in child2]

    return child1, child2

In [99]:
print("Allow duplicates:",one_point_crossover([1,2,3,4,5],[5,4,3,2,1]))
print("No duplicates:",one_point_crossover_no_duplicates([1,2,3,4,5],[5,4,3,2,1]))

Allow duplicates: ([1, 2, 3, 2, 1], [5, 4, 3, 4, 5])
No duplicates: ([1, 2, 5, 4, 3], [5, 4, 1, 2, 3])


In [100]:
def two_point_crossover(first,second):
    first_point = np.random.randint(0,len(first)-1)
    second_point = np.random.randint(first_point+1,len(first))

    print("Point 1:",first_point)
    print("Point 2:",second_point)
    return first[:first_point] + second[first_point:second_point] + first[second_point:], second[:first_point] + first[first_point:second_point] + second[second_point:]


def two_point_crossover_no_duplicates(first, second):
    first_point = np.random.randint(0, len(first) - 1)
    second_point = np.random.randint(first_point + 1, len(first))

    print("Point 1:",first_point)
    print("Point 2:",second_point)
    # Initialize children with None
    child1 = [None] * len(first)
    child2 = [None] * len(first)

    # Copy middle segments
    child1[first_point:second_point] = second[first_point:second_point]
    child2[first_point:second_point] = first[first_point:second_point]

    # Helper function to fill remaining positions
    def fill_child(child, parent):
        parent_idx = 0
        for i in range(len(child)):
            if child[i] is None:
                while parent[parent_idx] in child:
                    parent_idx += 1
                child[i] = parent[parent_idx]
                parent_idx += 1

    fill_child(child1, first)
    fill_child(child2, second)

    return child1, child2

In [101]:
print("Allow duplicates:",two_point_crossover([1,2,3,4,5],[5,4,3,2,1]))
print("No duplicates:",two_point_crossover_no_duplicates([1,2,3,4,5],[5,4,3,2,1]))

Point 1: 3
Point 2: 4
Allow duplicates: ([1, 2, 3, 2, 5], [5, 4, 3, 4, 1])
Point 1: 0
Point 2: 4
No duplicates: ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5])


## Uniform Crossover
Getting a gene from one of the parents randomly!

In [102]:
def uniform_crossover(first,second):
    child1 = []
    child2 = []
    for gene1, gene2 in zip(first, second):
        if np.random.rand() < 0.5:
            child1.append(gene1)
            child2.append(gene2)
        else:
            child1.append(gene2)
            child2.append(gene1)
    return child1, child2


In [103]:
uniform_crossover([1, 2, 3, 4, 5], [6,7,8,9,10])

([6, 2, 8, 4, 10], [1, 7, 3, 9, 5])

## Duplicates

One important property to consider in the homework assignments is whether the crossover operator can produce offspring without duplicates. In the Traveling Salesperson Problem (TSP), each city must appear exactly once in the resulting route. Using a crossover operator that does not preserve this constraint can produce invalid solutions, potentially hindering the genetic algorithm's performance.

Two crossover operators that preserve this property are Partially Mapped Crossover (PMX) and Cycle Crossover (CX). Both maintain the permutation structure of the solutions, ensuring that each city appears exactly once in the offspring. This makes them particularly useful when solving the homework assignment.

## Partially-mapped Crossover

**Main idea:** Exchange a segment between parents and use a mapping between the exchanged values to resolve duplicates.

PMX also works with permutations, but instead of identifying cycles, it selects two crossover points. One parent's segment is copied into the child, while the other parent's genes fill the remaining positions. Conflicts are resolved using the mapping between the two segments.

## Cycle Crossover

**Main idea:** Keep the positions of genes from one parent for a complete cycle of positions, and use the other parent to fill the remaining positions.

CX identifies cycles by following how the positions of values correspond between the two parents. Each position in a child gets its value from the parent assigned to that cycle. This guarantees a valid permutation.

## 2.3 Mutation

![image.png](images/mutations.png)

In [104]:
def swap_mutation(chromosome):
    first_point = np.random.randint(0,len(chromosome)-1)
    second_point = np.random.randint(first_point+1,len(chromosome))
    
    #print("Point 1:",first_point)
    #print("Point 2:",second_point)
    temp = chromosome[first_point]
    chromosome[first_point] = chromosome[second_point]
    chromosome[second_point] = temp

    return chromosome

def insert_mutation(chromosome):
    first_point = np.random.randint(0,len(chromosome)-1)
    second_point = np.random.randint(first_point+1,len(chromosome))
    
    print("Point 1:",first_point)
    print("Point 2:",second_point)
    value = chromosome.pop(first_point)
    chromosome.insert(second_point,value)

    return chromosome

def reverse_mutation(chromosome):
    first_point = np.random.randint(0,len(chromosome)-1)
    second_point = np.random.randint(first_point+1,len(chromosome))
    
    print("Point 1:",first_point)
    print("Point 2:",second_point)

    return chromosome[:first_point] + list(reversed(chromosome[first_point:second_point])) + chromosome[second_point:]

In [105]:
swap_mutation([1,2,3,4,5])

[1, 2, 4, 3, 5]

In [106]:
insert_mutation([1,2,3,4,5,6,7,8,9,10])

Point 1: 6
Point 2: 8


[1, 2, 3, 4, 5, 6, 8, 9, 7, 10]

In [107]:
reverse_mutation([1,2,3,4,5,6,7,8,9,10])

Point 1: 3
Point 2: 4


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

## N-Queens With Genetic Algorithm

In [110]:
def random_initialization(population_size, chromosome_length):
    population = []
    for _ in range(population_size):
        # This ensures that each queen is placed in a unique row and column
        chromosome = np.random.permutation(chromosome_length).tolist()
        population.append(chromosome)
    return population


def fitness_function(chromosome):
    fitness = 0
    n = len(chromosome)
    for col1 in range(n):
        for col2 in range(col1 + 1, n):
            row1, row2 = chromosome[col1], chromosome[col2]
            if row1 == row2:
                fitness += 1
            if abs(row1 - row2) == abs(col1 - col2):
                fitness += 1

    return n*(n-1)//2 - fitness


def genetic_algorithm(number_of_queens, population_size, generations, mutation_rate):
    population = random_initialization(population_size, number_of_queens)

    for generation in range(generations):

        # Evaluate fitness
        fitness_scores = [fitness_function(chromosome)
                          for chromosome in population]
        best_fitness = max(fitness_scores)
        print(f"Generation {generation}: Best fitness = {best_fitness} of {number_of_queens*(number_of_queens-1)//2}")
        # It is good to add elitism here

        if best_fitness == number_of_queens*(number_of_queens-1)//2:
            print("Solution found!")
            return population[fitness_scores.index(best_fitness)]

        # Selection
        selected = tournament_selection(
            population, fitness_scores, tournament_size=20, num_selections=population_size)

        # Crossover
        next_generation = []
        for i in range(1, population_size, 2):
            parent1 = selected[i]
            parent2 = selected[i-1]
            child1, child2 = one_point_crossover_no_duplicates(
                parent1, parent2)
            next_generation.extend([child1, child2])

        # Mutation
        for i in range(population_size):
            if np.random.rand() < mutation_rate:
                next_generation[i] = swap_mutation(next_generation[i])

        population = next_generation

    print("Solution not found!")

    return population[fitness_scores.index(best_fitness)]

In [123]:
genetic_algorithm(number_of_queens=30, population_size=500,
                  generations=100, mutation_rate=0.2)

Generation 0: Best fitness = 427 of 435
Generation 1: Best fitness = 428 of 435
Generation 2: Best fitness = 428 of 435
Generation 3: Best fitness = 429 of 435
Generation 4: Best fitness = 429 of 435
Generation 5: Best fitness = 430 of 435
Generation 6: Best fitness = 431 of 435
Generation 7: Best fitness = 431 of 435
Generation 8: Best fitness = 433 of 435
Generation 9: Best fitness = 433 of 435
Generation 10: Best fitness = 433 of 435
Generation 11: Best fitness = 433 of 435
Generation 12: Best fitness = 433 of 435
Generation 13: Best fitness = 433 of 435
Generation 14: Best fitness = 433 of 435
Generation 15: Best fitness = 433 of 435
Generation 16: Best fitness = 433 of 435
Generation 17: Best fitness = 433 of 435
Generation 18: Best fitness = 433 of 435
Generation 19: Best fitness = 433 of 435
Generation 20: Best fitness = 433 of 435
Generation 21: Best fitness = 433 of 435
Generation 22: Best fitness = 433 of 435
Generation 23: Best fitness = 433 of 435
Generation 24: Best fitnes

[24,
 9,
 1,
 12,
 19,
 6,
 23,
 18,
 22,
 28,
 16,
 2,
 4,
 26,
 8,
 17,
 11,
 3,
 0,
 15,
 20,
 25,
 27,
 21,
 7,
 10,
 29,
 14,
 5,
 13]

## Homework: TSP

Implement a Genetic Algorithm (GA) for one of the following two problems:

Solve the [Traveling Salesman Problem](https://en.wikipedia.org/wiki/Travelling_salesman_problem) using a Genetic Algorithm.

The goal is to find the shortest possible route that visits every point exactly once. There is a path between every pair of points. The starting point does not matter, and the route does not need to return to the starting point.

#### Example

![image.png](images/TSP.png)


### Input

One of the following:

1. An integer N (N ≤ 100) - generate N random points in the plane.

2. A dataset name (a string containing letters and optionally digits), followed by the number of cities and lines in the format:

`<city_name> X Y`

### Output

* At least 10 values (one per line) representing the best route length found in the population so far: the first generation, at least eight intermediate generations, and the final generation.

* An empty line.

* The best solution found:

  1. For N random points: Output only the final route length (a single value), which must equal the last value in the block above.

  2. For a named dataset: Output one line containing the route in the format `CityA -> CityB -> ...`, followed by a line containing the final route length.

The final route length must match the last value in the block above and the length recalculated from the route.

Sample Input:

```
UK12
12
Aberystwyth 0.190032E-03 -0.285946E-03
...
Stratford 217.343 -447.089
```

Sample Output:

```
2426.8086
...
1595.7385

Aberystwyth -> Inverness -> Nottingham -> Glasgow -> Edinburgh -> London -> Stratford -> Exeter -> Liverpool -> Oxford -> Brighton -> Newcastle
1595.7385
```


## Homework: KP



Solve the [Knapsack Problem](https://en.wikipedia.org/wiki/Knapsack_problem) using a Genetic Algorithm (GA).

The goal is to select a subset of items with the maximum possible total value without exceeding the knapsack's capacity.

![image.png](images/KnapsackProblem.png)

### Input

* One line containing M N — the knapsack's capacity and the number of items.

* The next N lines each contain two integers:

  `mi ci` — the weight and value of the i-th item.

### Output

* At least 10 values (one per line) representing the best value found in the population so far: the first generation, at least eight intermediate generations, and the final generation.

* An empty line.

* The final maximum value (a single value), equal to the last value in the block above.

### Sample Input

```
5000 24
90 150
...
2000 150
```

### Sample Output

```
1010
...
1130

1130
```


### Optimizations

- Appropriate crossover function
- Parallel execution
- Selection function with the right amount of selection pressure
- These problems require extensive experimentation to find the most effective parameter values. This includes choosing suitable crossover, selection, and mutation strategies, as well as tuning their respective probabilities, parameters, and other configuration settings.
